## Consistency Evaluation

In this small notebook, we calculate $\mu$ and $\sigma$ of five consecutive evaluations of the advanced Target System B. We used the *attentive* persona to generate all dialogues. Each round was conducted with the same initial question and starting questions. The dialogue generation configuration was $n=5, \alpha=2, \beta=3,T_{\text{max}}=3,N=20$, no caching was used aside the initial root caching which assures comparability of the results.

In [13]:
import json
import glob
import os
import statistics

def analyze_all_evaluations(base_folder):
    patterns = [
        os.path.join(base_folder, "**", "*evaluation_single_turn.json"),
        os.path.join(base_folder, "**", "*evaluation_multi_turn.json")
    ]
    
    files = []
    for pattern in patterns:
        files.extend(glob.glob(pattern, recursive=True))
        
    files = sorted(list(set(files)))

    if not files:
        print(f"Keine passenden Dateien im Pfad '{base_folder}' gefunden.")
        return

    print(f"Insgesamt {len(files)} Dateien für die Analyse gefunden.\n")

    all_values = {
        "[multi_hop] correctness_aspect_critic": [],
        "[multi_hop] faithfulness_aspect_critic": [],
        "[multi_hop] context_precision_critic": [],
        "[multi_hop] context_recall_critic": [],
        "[single_hop] correctness_aspect_critic": [],
        "[single_hop] faithfulness_aspect_critic": [],
        "[single_hop] context_precision_critic": [],
        "[single_hop] context_recall_critic": [],
        "[multi_turn] average_forgetfulness_aspect_critic": [],
        "[multi_turn] average_context_retention_aspect_critic": []
    }

    single_turn_keys = [
        ("multi_hop_evaluation", "correctness_aspect_critic"),
        ("multi_hop_evaluation", "faithfulness_aspect_critic"),
        ("multi_hop_evaluation", "context_precision_critic"),
        ("multi_hop_evaluation", "context_recall_critic"),
        ("single_hop_evaluation", "correctness_aspect_critic"),
        ("single_hop_evaluation", "faithfulness_aspect_critic"),
        ("single_hop_evaluation", "context_precision_critic"),
        ("single_hop_evaluation", "context_recall_critic"),
    ]

    for file_path in files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                if "single_turn" in os.path.basename(file_path):
                    for eval_type, metric_name in single_turn_keys:
                        try:
                            val = data[eval_type]["average_scores"][metric_name]
                            dict_key = f"[{eval_type.replace('_evaluation', '')}] {metric_name}"
                            all_values[dict_key].append(val)
                        except KeyError:
                            pass 
                
                elif "multi_turn" in os.path.basename(file_path):
                    for mt_key in ["average_forgetfulness_aspect_critic", "average_context_retention_aspect_critic"]:
                        if mt_key in data:
                            all_values[f"[multi_turn] {mt_key}"].append(data[mt_key])
                            
        except Exception as e:
            print(f"Fehler beim Lesen der Datei {os.path.basename(file_path)}: {e}")

    # --- print results ---
    print("="*80)
    print(" GESAMTAUSWERTUNG: STANDARDABWEICHUNG & MIN-MAX-SCHWANKE")
    print("="*80)

    for metric_name, vals in all_values.items():
        print(f"{metric_name}:")
        
        if len(vals) > 1:
            std_val = statistics.stdev(vals)
            min_val = min(vals)
            max_val = max(vals)
            span_val = max_val - min_val 
            
            print(f"   Datenpunkte erfasst: {len(vals)}")
            print(f"   Standardabweichung (SD): {std_val:.4f}")
            print(f"   Minimum: {min_val:.4f}  |  Maximum: {max_val:.4f}  |  Schwanke (Spanne): {span_val:.4f}")
        elif len(vals) == 1:
            print(f"   Nur 1 Datenpunkt (Wert: {vals[0]:.4f}). Varianz nicht berechenbar.")
        else:
            print("   Keine Daten gefunden.")
        print("-" * 80)

if __name__ == "__main__":
    folder_path = "evaluations" 
    
    analyze_all_evaluations(folder_path)

Insgesamt 10 Dateien für die Analyse gefunden.

 GESAMTAUSWERTUNG: STANDARDABWEICHUNG & MIN-MAX-SCHWANKE
[multi_hop] correctness_aspect_critic:
   Datenpunkte erfasst: 5
   Standardabweichung (SD): 0.0427
   Minimum: 0.8500  |  Maximum: 0.9524  |  Schwanke (Spanne): 0.1024
--------------------------------------------------------------------------------
[multi_hop] faithfulness_aspect_critic:
   Datenpunkte erfasst: 5
   Standardabweichung (SD): 0.0236
   Minimum: 0.8322  |  Maximum: 0.8875  |  Schwanke (Spanne): 0.0554
--------------------------------------------------------------------------------
[multi_hop] context_precision_critic:
   Datenpunkte erfasst: 5
   Standardabweichung (SD): 0.0564
   Minimum: 0.8849  |  Maximum: 1.0000  |  Schwanke (Spanne): 0.1151
--------------------------------------------------------------------------------
[multi_hop] context_recall_critic:
   Datenpunkte erfasst: 5
   Standardabweichung (SD): 0.0116
   Minimum: 0.6984  |  Maximum: 0.7250  |  Schwan